In [2]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/database/GSE211956/GSE211956_SC.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/GSE211956/GSE211956_ST_P2.h5ad" \
    --n_epochs 300 \
    --resolution 2 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 100 \
    --latent_dim 256 \
    --output_dir ../deconv_results/GSE211956 \
    --aggregation_method mean
 #   --precomputed_marker_file "../deconv_results/CID44971/final_genes.txt"

Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 2.0
   Batch size: 256
   Epochs: 300
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/database/GSE211956/GSE211956_SC.h5ad
   SC shape: (14299, 19915)
   SC avg counts/cell: 2514.9 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/database/GSE211956/GSE211956_ST_P2.h5ad
   ST shape: (1935, 33538)
   Common genes: 19899
   SC final: (14299, 19899)
   ST final: (1935, 19899)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 35 clusters
Marker genes per cluster:
   0: 61 -> 61 (after Lasso)
   1: 34 -> 34 (after Lasso)
   10: 93 -> 93 (after Lasso)
   11: 76 -> 76 (after Lasso)
   12: 100 -> 100 (after Lasso)
 

VAE Training:  75%|███████▌  | 226/300 [08:13<02:41,  2.18s/epoch, Train=1079.9223, Recon=1076.3438, KL=35.7840, MMD=0.0165, Test=1120.6122]



⚠️ Early stopping triggered at epoch 227/300
   Best test loss: 1120.3086, Current test loss: 1120.6122
   No improvement for 20 consecutive epochs
📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (14293, 990)
   Number of clusters: 34
   Computed embeddings shape: (14293, 256)
Computing cluster centers with mean aggregation...
      Cluster 0: 899 cells (mean aggregation)
      Cluster 1: 865 cells (mean aggregation)
      Cluster 2: 577 cells (mean aggregation)
      Cluster 3: 512 cells (mean aggregation)
      Cluster 4: 489 cells (mean aggregation)
      Cluster 5: 472 cells (mean aggregation)
      Cluster 6: 468 cells (mean aggregation)
      Cluster 7: 453 cells (mean aggregation)
      Cluster 8: 448 cells (mean aggregation)
      Cluster 9: 440 cells (mean aggregation)
      Cluster 10: 439 cells (mean aggregation)
      Cluster 11: 383 cells (mean aggregation)
      Cluster 12: 854 cells (mean aggregation)
      Cluster 13: 356 cells (mean agg

In [3]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/database/GSE211956/GSE211956_ST_P2.h5ad" \
    --stage1_model_path "../deconv_results/GSE211956/final_vae.pth" \
    --output_dir "../deconv_results/GSE211956/" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 1e-4 \
    --loss_lambda_mse 0.05 \
    --loss_lambda_cosine 2 \
    --loss_lambda_pearson 2 \
    --loss_lambda_reg 0.1 \
    --loss_lambda_sparse 0.01 \
    --loss_lambda_proportion 0.1 \
    --k_spatial 10 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 500 \
    --weight_threshold 0.05

Sample name: GSE211956_ST_P2
Stage 1 model: ../deconv_results/GSE211956/final_vae.pth
Output directory: ../deconv_results/GSE211956/
Device: cpu
Weight threshold: 0.05
Loading pretrained VAE Encoder...
   VAE architecture: 990 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 14293 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/GSE211956/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([34, 256])
   Cluster expressions (marker): torch.Size([34, 990])
   Cluster expressions (all genes): 34 × 19899
   Loaded celltype mapping: 34 clusters
   Average cell counts: 2514.9
Loaded all genes list: 19899 genes
VAE Encoder loaded: 990 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '4', '5', '6', '7', '8', '9']
Marker genes: 990
Using Stage 1 cluster centers and expressions

GAT Training:  37%|███▋      | 185/500 [10:45<18:18,  3.49s/epoch, Total=22.1468, Pearson=0.4329, MSE=412.4562, Cosine=0.2452, P_pat=12, M_pat=10, C_pat=12]



⚠️ Early stopping triggered at epoch 186/500
   All three core losses stopped improving:
      Pearson: best=0.4317, current=0.4329, no improvement for 12 epochs
      MSE: best=411.9604, current=412.4562, no improvement for 10 epochs
      Cosine: best=0.2445, current=0.2452, no improvement for 12 epochs
Evaluating model results...
Applying weight threshold: 0.05
   Non-zero elements: 38700 -> 5413 (8.2%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/GSE211956//GSE211956_ST_P2_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['Fibro1 (EIF4A3, STAR)', 'Endothelial cells', 'Tumour cells', 'Myofibroblasts', 'Fibro2 (RBP1, DCN)']. Merging corresponding cluster columns by summing weights.
   Columns before: 34, after merge: 11
   Saved cell composition (celltype): ../deconv_results/GSE211956//GSE211956_ST_P2_cell_composition.csv
   Saved cluster 